# Task 03 – Human & Car Detection with Counting
## Inference Pipeline with Bounding Box Visualization

This notebook demonstrates:
1. Running detection on single images
2. Bounding box drawing (green=person, orange=car)
3. Human count overlay
4. Batch processing on validation set
5. Count statistics and confidence distribution

In [ ]:
import sys, os
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

from src.detect import run_inference, count_objects, draw_detections, process_folder

# ── Paths ──
DATA_ROOT   = r'D:\antlings-drone-detection\dataset\VisDrone_Dataset'
VAL_IMGS    = str(Path(DATA_ROOT) / 'VisDrone2019-DET-val' / 'images')
TEST_IMGS   = str(Path(DATA_ROOT) / 'VisDrone2019-DET-test-dev' / 'images')
WEIGHTS     = '../runs/detect/visdrone_yolov8n/weights/best.pt'
OUT_DIR     = '../outputs/images'

os.makedirs(OUT_DIR, exist_ok=True)

model = YOLO(WEIGHTS)
print('Model loaded.')
print(f'Val images : {VAL_IMGS}')

In [ ]:
# Single image inference demo
test_img = sorted(Path(VAL_IMGS).glob('*.jpg'))[0]
frame    = cv2.imread(str(test_img))

detections = run_inference(model, frame)
counts     = count_objects(detections)
annotated  = draw_detections(frame, detections, counts)

print(f'Image      : {test_img.name}')
print(f'Detections : {len(detections)}')
print(f'Persons    : {counts["person"]}')
print(f'Cars       : {counts["car"]}')

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'Detection — Person: {counts["person"]}  Car: {counts["car"]}', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_single_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/images/03_single_detection.png')

In [ ]:
# Detection grid — 9 validation images
val_imgs = sorted(Path(VAL_IMGS).glob('*.jpg'))[:9]

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('Detection Results – VisDrone Validation Set', fontsize=14, fontweight='bold')

for idx, img_path in enumerate(val_imgs):
    frame  = cv2.imread(str(img_path))
    dets   = run_inference(model, frame)
    counts = count_objects(dets)
    ann    = draw_detections(frame, dets, counts)

    ax = axes[idx // 3][idx % 3]
    ax.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Person: {counts["person"]}  Car: {counts["car"]}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_detection_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/images/03_detection_grid.png')

In [ ]:
# Batch processing on validation set
results = process_folder(model, VAL_IMGS, OUT_DIR)

persons = [r['person_count'] for r in results]
cars    = [r['car_count']    for r in results]

print(f'\nBatch Summary:')
print(f'  Images processed  : {len(results)}')
print(f'  Total persons     : {sum(persons)}')
print(f'  Total cars        : {sum(cars)}')
print(f'  Avg persons/image : {np.mean(persons):.1f}')
print(f'  Avg cars/image    : {np.mean(cars):.1f}')
print(f'  Max persons       : {max(persons)}')
print(f'  Avg inference     : {np.mean([r["inference_ms"] for r in results]):.1f}ms')

In [ ]:
# Count distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Object Count Distribution – VisDrone Validation Set', fontsize=13, fontweight='bold')

axes[0].hist(persons, bins=25, color='#00C853', edgecolor='none', alpha=0.85)
axes[0].set_title(f'Person Count per Image (total={sum(persons):,})')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(np.mean(persons), color='red', linestyle='--', label=f'Mean: {np.mean(persons):.1f}')
axes[0].legend()

axes[1].hist(cars, bins=25, color='#FF6D00', edgecolor='none', alpha=0.85)
axes[1].set_title(f'Car Count per Image (total={sum(cars):,})')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(np.mean(cars), color='red', linestyle='--', label=f'Mean: {np.mean(cars):.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/03_count_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/images/03_count_histogram.png')

## Counting Logic

Simple per-frame counting:

```python
def count_objects(detections):
    return {
        'person': sum(1 for d in detections if d['cls_id'] == 0),
        'car':    sum(1 for d in detections if d['cls_id'] == 1),
    }
```

- **Images**: per-frame count displayed as overlay
- **Video with tracking (Task 04)**: unique track IDs counted to avoid double-counting

### Task 03 Complete
- Detection running on VisDrone val split
- Bounding boxes with confidence scores displayed
- Person and car counts overlaid on each frame
- Results saved to `outputs/images/`